# 05 Multimodal Model Training

This notebook trains and evaluates a multimodal regression model using text and image embeddings. It recreates the text baseline preprocessing, verifies alignment with image features, fuses modalities, and compares results to the text-only baseline.

## 1. Setup and Imports

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    xgboost_available = True
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    xgboost_available = False

print("Setup complete")
print("XGBoost available:", xgboost_available)

Setup complete
XGBoost available: True


## 2. Load data and verify alignment

In [3]:
# Load data files without modifying them
sample_path = Path("../data/sample_5000.csv")
image_features_path = Path("../data/image_features.npy")

sample_df = pd.read_csv(sample_path)
image_features = np.load(image_features_path)

print("Dataset shape:", sample_df.shape)
print("Image feature shape:", image_features.shape)

if sample_df.shape[0] != image_features.shape[0]:
    raise ValueError(
        f"Sample count mismatch: {sample_df.shape[0]} rows in sample_5000.csv vs {image_features.shape[0]} rows in image_features.npy."
    )

Dataset shape: (5000, 4)
Image feature shape: (5000, 1280)


## 3. Recreate text features using the baseline preprocessing

In [4]:
# Match the preprocessing used in 03_text_baseline.ipynb
sample_df["catalog_content"] = (
    sample_df["catalog_content"].astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_text = vectorizer.fit_transform(sample_df["catalog_content"])
print("TF-IDF text feature shape:", X_text.shape)

TF-IDF text feature shape: (5000, 20000)


## 4. Combine text and image embeddings

In [5]:
X_multimodal = hstack([X_text, csr_matrix(image_features)])
print("Combined multimodal feature shape:", X_multimodal.shape)

Combined multimodal feature shape: (5000, 21280)


## 5. Train/test split

In [6]:
# Use the same target transformation as the text baseline
y = np.log1p(sample_df["price"])

X_text_train, X_text_test, X_multi_train, X_multi_test, y_train, y_test = train_test_split(
    X_text,
    X_multimodal,
    y,
    test_size=0.2,
    random_state=42,
)

print("Text train shape:", X_text_train.shape)
print("Text test shape:", X_text_test.shape)
print("Multimodal train shape:", X_multi_train.shape)
print("Multimodal test shape:", X_multi_test.shape)

Text train shape: (4000, 20000)
Text test shape: (1000, 20000)
Multimodal train shape: (4000, 21280)
Multimodal test shape: (1000, 21280)


## 6. Text-only baseline model

In [7]:
if xgboost_available:
    baseline_model = XGBRegressor(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
    )
    baseline_name = "XGBRegressor"
else:
    baseline_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        random_state=42,
        n_jobs=-1,
    )
    baseline_name = "RandomForestRegressor (fallback)"

print("Baseline model:", baseline_name)
baseline_model.fit(X_text_train, y_train)

baseline_preds_log = baseline_model.predict(X_text_test)
baseline_preds = np.expm1(baseline_preds_log)
actual_prices = np.expm1(y_test)

baseline_mae = mean_absolute_error(actual_prices, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(actual_prices, baseline_preds))
baseline_r2 = r2_score(actual_prices, baseline_preds)

print("Text baseline MAE:", baseline_mae)
print("Text baseline RMSE:", baseline_rmse)
print("Text baseline R²:", baseline_r2)

Baseline model: XGBRegressor
Text baseline MAE: 14.035422466478346
Text baseline RMSE: 33.35725109013283
Text baseline R²: 0.08575599198447204


## 7. Multimodal regression model

In [8]:
if xgboost_available:
    multimodal_model = XGBRegressor(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
    )
    multimodal_name = "XGBRegressor"
else:
    multimodal_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        random_state=42,
        n_jobs=-1,
    )
    multimodal_name = "RandomForestRegressor (fallback)"

print("Multimodal model:", multimodal_name)
multimodal_model.fit(X_multi_train, y_train)

multi_preds_log = multimodal_model.predict(X_multi_test)
multi_preds = np.expm1(multi_preds_log)

multi_mae = mean_absolute_error(actual_prices, multi_preds)
multi_rmse = np.sqrt(mean_squared_error(actual_prices, multi_preds))
multi_r2 = r2_score(actual_prices, multi_preds)

print("Multimodal MAE:", multi_mae)
print("Multimodal RMSE:", multi_rmse)
print("Multimodal R²:", multi_r2)

Multimodal model: XGBRegressor
Multimodal MAE: 14.05552897810936
Multimodal RMSE: 33.194582977586506
Multimodal R²: 0.09465095344193264


## 8. Results comparison

In [9]:
comparison_df = pd.DataFrame(
    {
        "Model": ["Text Baseline", "Multimodal"],
        "MAE": [baseline_mae, multi_mae],
        "RMSE": [baseline_rmse, multi_rmse],
        "R²": [baseline_r2, multi_r2],
    }
)

comparison_df.set_index("Model", inplace=True)
comparison_df

,MAE,RMSE,R²
Model,,,
Text Baseline,14.035422,33.357251,0.085756
Multimodal,14.055529,33.194583,0.094651


## 9. Why image embeddings help

Image embeddings capture visual signals such as color, shape, and product presentation that are not present in text alone. These visual representations can provide complementary price cues when fused with text features.

## 10. How multimodal learning works

This notebook uses early fusion: text and image embeddings are concatenated into a single feature matrix. The regression model trains on the combined modality representation, allowing it to learn from both text and image signals at once.

## 11. Interpretation of evaluation metrics

- **MAE** measures the average absolute prediction error in price.
- **RMSE** penalizes larger errors more strongly and is sensitive to outliers.
- **R²** shows how much variance in price the model explains.

Lower MAE/RMSE is better. Higher R² is better.

## 12. Did multimodal improve over the text baseline?

Compare the two rows above. If the multimodal row has lower MAE/RMSE and higher R², then adding image embeddings improved the model.